# Day 93: Causal Trees, Causal Forests and Honest Inference

*Beginner to Ninja: Math for Causal Inference*

Trees partition the covariate space into leaves and estimate an effect within each. Choosing splits to maximise effect differences and then estimating effects on the same data exaggerates heterogeneity and breaks confidence intervals. Honesty (one sample to choose the splits, another to estimate) fixes this. Averaging many honest trees gives a causal forest, whose predictions are adaptive-kernel estimates with valid asymptotic inference.

**Learning outcomes**

- Derive the causal-tree splitting criterion and relate it to the variance of leaf effects.
- Explain adaptive selection bias and prove that honest leaf estimates are unbiased.
- Express a causal forest prediction as a weighted local estimate with forest weights.
- Test for heterogeneity with the best linear predictor and sorted group effects (GATES).

## Theory

### A causal tree

Partition $\mathcal X$ into leaves $\ell$. In a randomised experiment, estimate each leaf effect by a difference in means:

$$
\hat\tau_\ell = \bar Y_{1,\ell} - \bar Y_{0,\ell}.
$$

Athey and Imbens choose splits to maximise $\sum_\ell n_\ell\hat\tau_\ell^2$, which (since $\sum_\ell n_\ell\hat\tau_\ell$ is fixed) is the same as maximising the spread of leaf effects, with a penalty for leaf-level variance.

### Adaptive selection bias

The split is chosen *because* the estimated effects differ. Re-using the same data to estimate leaf effects then overstates differences and produces confidence intervals that undercover: the winner's curse.

### Honesty

Split the sample: sample A chooses the partition, sample B estimates the leaf effects. Given the partition (a function of A only), B's estimates are ordinary difference-in-means estimates with valid standard errors.

### Causal forests

Average $B$ honest trees grown on subsamples with random feature choice. The prediction at $x$ uses **forest weights**

$$
\alpha_i(x) = \frac1B\sum_{b = 1}^B\frac{\mathbf 1\{X_i \in L_b(x)\}}{|L_b(x)|}, \qquad \sum_i\alpha_i(x) = 1,
$$

and solves a locally weighted moment equation, for example the R-learner residual regression $\sum_i\alpha_i(x)(\tilde Y_i - \tau\tilde D_i)\tilde D_i = 0$ (generalised random forests, Athey, Tibshirani and Wager). With honest subsampled trees, $\hat\tau(x)$ is asymptotically normal with a computable variance.

### Is the heterogeneity real?

On held-out data with AIPW scores $\hat\varphi_i$ (Day 92):

- **BLP test:** regress $\hat\varphi_i$ on $1$ and $\hat\tau(X_i) - \overline{\hat\tau}$. A slope significantly above zero means $\hat\tau$ captures real heterogeneity; a slope near 1 means it is well calibrated.
- **GATES:** sort units into quintiles of $\hat\tau$ and report the average of $\hat\varphi$ within each with SEs.

## Part 1 · Warm-up

### Exercise 1 · A leaf effect

In a leaf, 40 treated patients have mean adherence 72 and 38 controls have mean 65. Estimate the leaf effect.

<details><summary>Show solution</summary>

**Answer:** $\hat\tau_\ell = 72 - 65 = 7$ points.

</details>

*Your work:*

### Exercise 2 · The splitting criterion

A parent leaf of 200 units with $\hat\tau = 2$ could split into (120 units, $\hat\tau = 1$) and (80 units, $\hat\tau = 3.5$). Compute the criterion before and after the split.

<details><summary>Show solution</summary>

**Step 1.** Before: $200\times4 = 800$.

**Step 2.** After: $120\times1 + 80\times12.25 = 120 + 980 = 1{,}100$.

**Answer:** the split raises the criterion by 300, so it is attractive (before any variance penalty).

</details>

*Your work:*

### Exercise 3 · Forest weights

Two trees. In tree 1, $x$'s leaf contains units $\{1, 2\}$; in tree 2, units $\{2, 3, 4, 5\}$. Compute $\alpha_i(x)$.

<details><summary>Show solution</summary>

**Step 1.** Tree 1 gives $1/2$ to units 1 and 2; tree 2 gives $1/4$ to units 2–5.

**Step 2.** Average: $\alpha_1 = 0.25$, $\alpha_2 = 0.375$, $\alpha_3 = \alpha_4 = \alpha_5 = 0.125$.

**Answer:** the weights sum to 1; unit 2, which shares a leaf with $x$ in both trees, counts most.

</details>

*Your work:*

### Exercise 4 · Honest split sizes

With 2,000 participants and a 50/50 honest split, how many observations estimate leaf effects, and what is the cost?

<details><summary>Show solution</summary>

**Step 1.** 1,000 estimate the effects; leaf SEs are about $\sqrt2$ times larger than with all data.

**Answer:** forests recover most of this loss by rotating roles across subsamples.

</details>

*Your work:*

### Exercise 5 · Reading a BLP

The BLP slope is 0.9 (SE 0.3). What do you conclude?

<details><summary>Show solution</summary>

**Step 1.** $t = 3$: significant heterogeneity captured by $\hat\tau$.

**Step 2.** A slope near 1: predictions are roughly calibrated (neither exaggerated nor dampened).

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Criterion equals spread of leaf effects

Show $\sum_\ell n_\ell\hat\tau_\ell^2 = n\bar\tau^2 + \sum_\ell n_\ell(\hat\tau_\ell - \bar\tau)^2$, where $\bar\tau = \sum_\ell n_\ell\hat\tau_\ell/n$.

<details><summary>Show solution</summary>

**Step 1.** Expand $\sum n_\ell(\hat\tau_\ell - \bar\tau)^2 = \sum n_\ell\hat\tau_\ell^2 - 2\bar\tau\sum n_\ell\hat\tau_\ell + n\bar\tau^2 = \sum n_\ell\hat\tau_\ell^2 - n\bar\tau^2$.

**Step 2.** Rearrange. $\blacksquare$ For fixed $\bar\tau$, maximising the criterion maximises the between-leaf variance of effects.

</details>

*Your work:*

### Exercise 7 · The winner's curse

Two candidate splits have estimated effect differences $Z_1, Z_2$ i.i.d. $N(0, 1)$ (no true heterogeneity). Show $E[\max(Z_1, Z_2)] = 1/\sqrt\pi$.

<details><summary>Show solution</summary>

**Step 1.** $\max(Z_1, Z_2) = \frac{Z_1 + Z_2}2 + \frac{|Z_1 - Z_2|}2$.

**Step 2.** $Z_1 - Z_2 \sim N(0, 2)$, so $E|Z_1 - Z_2| = \sqrt2\sqrt{2/\pi} = 2/\sqrt\pi$.

**Step 3.** $E[\max] = 0 + 1/\sqrt\pi \approx 0.56$. $\blacksquare$ The selected difference is biased upward even though the truth is zero; with many candidate splits, more so.

</details>

*Your work:*

### Exercise 8 · Honest estimates are unbiased

In a randomised experiment, let the partition $\Pi$ depend only on sample A. Show that for sample B, $E[\hat\tau_\ell \mid \Pi] = E[Y(1) - Y(0) \mid X \in \ell]$ (conditioning also on both arms being non-empty).

<details><summary>Show solution</summary>

**Step 1.** Sample B is independent of A, hence of $\Pi$; given $\Pi$, leaf $\ell$ is a fixed set.

**Step 2.** Within $\ell$, treatment in B is randomised, so $E[\bar Y_{1,\ell}] = E[Y(1) \mid X \in \ell]$ and likewise for controls.

**Step 3.** Subtract. $\blacksquare$ The usual difference-in-means SEs are therefore valid.

</details>

*Your work:*

### Exercise 9 · Forest weights sum to one

Show $\sum_i\alpha_i(x) = 1$ and that the forest average of leaf means equals $\sum_i\alpha_i(x)Y_i$.

<details><summary>Show solution</summary>

**Step 1.** For tree $b$: $\sum_i\mathbf 1\{X_i \in L_b(x)\}/|L_b(x)| = 1$; averaging over trees keeps the sum at 1.

**Step 2.** Tree $b$'s leaf mean is $\sum_i Y_i\mathbf 1\{X_i \in L_b\}/|L_b|$; averaging over $b$ and swapping the sums gives $\sum_i\alpha_i(x)Y_i$. $\blacksquare$

</details>

*Your work:*

### Exercise 10 · The BLP slope

Suppose $E[\varphi \mid X] = \tau(X)$ and $\hat\tau$ is fixed (fitted on other data). Show that the population slope of $\varphi$ on $\hat\tau - E\hat\tau$ is $\mathrm{Cov}(\tau, \hat\tau)/\mathrm{Var}(\hat\tau)$, which equals 1 if $\hat\tau = \tau$ and 0 if $\tau$ is constant.

<details><summary>Show solution</summary>

**Step 1.** Slope $= \mathrm{Cov}(\varphi, \hat\tau)/\mathrm{Var}(\hat\tau)$, and $\mathrm{Cov}(\varphi, \hat\tau) = \mathrm{Cov}(E[\varphi \mid X], \hat\tau) = \mathrm{Cov}(\tau, \hat\tau)$.

**Step 2.** If $\hat\tau = \tau$: ratio 1. If $\tau$ is constant: covariance 0. $\blacksquare$

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · Text reminders for medication adherence

A trial randomises 1,000 patients to text reminders; ten baseline covariates are available; the true effect is the same for everyone. A one-split causal tree is grown 300 times on fresh data. Compare adaptive and honest estimation.

<details><summary>Show solution</summary>

**Step 1.** Adaptive (same data): average gap between the two leaf effects 0.26 even though the truth is 0; leaf CIs cover the true effect only 83% of the time.

**Step 2.** Honest (split on half, estimate on the other): coverage 95%.

**Step 3.** The honest gap (0.21) is pure noise, of the size expected from two independent estimates on 250 units each; its CIs say so.

**Answer:** without honesty, the tree reports heterogeneity that is not there, with overconfident intervals.

</details>

*Your work:*

### Exercise 12 · Real heterogeneity

Now the reminder effect is 1 for patients with low baseline adherence score $x_1 \le 0$ and 2 otherwise. The honest tree picks $x_1$ in 58% of replications; when it does, its leaf estimates average 1.06 and 1.94. Interpret.

<details><summary>Show solution</summary>

**Step 1.** When the right variable is found, honest estimates are close to the truth.

**Step 2.** With only 500 units to choose splits, the tree often chooses a noise variable; forests average many trees to stabilise this.

</details>

*Your work:*

### Exercise 13 · GATES for a microfinance programme

Held-out AIPW scores averaged by quintile of predicted effect on business profits (in \$) are 2, 5, 9, 14, 30 with SEs of about 6. What do you report?

<details><summary>Show solution</summary>

**Step 1.** Effects rise across quintiles; top minus bottom $= 28$ with SE $\approx\sqrt{36 + 36} = 8.5$, so $t \approx 3.3$.

**Answer:** the model finds real heterogeneity; gains are concentrated in the top quintile, so targeting could raise impact.

</details>

*Your work:*

### Exercise 14 · Avoiding subgroup fishing

A team ran 40 subgroup analyses of a class-size reduction and highlights the three significant ones. How do forests plus BLP/GATES avoid this problem?

<details><summary>Show solution</summary>

**Step 1.** The forest searches over subgroups in one algorithm, with honesty or cross-fitting separating search from estimation.

**Step 2.** The BLP/GATES tests are pre-specified, single tests of heterogeneity on held-out data, so there is no multiple-testing inflation from cherry-picking.

</details>

*Your work:*

### Exercise 15 · Explaining honesty

Explain honesty to a hospital board in two sentences.

<details><summary>Show solution</summary>

**Step 1.** "We use half the patients to look for groups where the reminders seem to work differently, and the other half to measure how well they actually work in those groups."

**Step 2.** "Because the second half played no part in choosing the groups, its results are not flattered by the search."

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np

rng = np.random.default_rng(93)
def dm(Y, D):
    y1, y0 = Y[D == 1], Y[D == 0]
    return y1.mean() - y0.mean(), np.sqrt(y1.var(ddof=1) / len(y1) + y0.var(ddof=1) / len(y0))
def best_split(X, Y, D, minleaf=50):           # maximise sum_l n_l * tau_l^2
    best = (-np.inf, None, None)
    for j in range(X.shape[1]):
        for c in np.quantile(X[:, j], np.linspace(.1, .9, 17)):
            L = X[:, j] <= c
            if min(L.sum(), (~L).sum()) < minleaf: continue
            crit = L.sum() * dm(Y[L], D[L])[0] ** 2 + (~L).sum() * dm(Y[~L], D[~L])[0] ** 2
            if crit > best[0]: best = (crit, j, c)
    return best[1:]
def run(n=1000, het=0.0):
    X = rng.normal(size=(n, 10)); D = rng.binomial(1, .5, n)
    Y = X[:, 1] + (1 + het * (X[:, 0] > 0)) * D + rng.normal(size=n)
    j, c = best_split(X, Y, D); L = X[:, j] <= c
    adaptive = [dm(Y[L], D[L]), dm(Y[~L], D[~L])]
    h = rng.permutation(n) < n // 2                           # honest: split on h, estimate on ~h
    j2, c2 = best_split(X[h], Y[h], D[h]); L2 = X[~h, j2] <= c2
    honest = [dm(Y[~h][L2], D[~h][L2]), dm(Y[~h][~L2], D[~h][~L2])]
    return adaptive, honest, j2

gA, gH, cA, cH = [], [], [], []
for _ in range(300):
    a, b, _ = run()
    gA.append(abs(a[0][0] - a[1][0])); gH.append(abs(b[0][0] - b[1][0]))
    cA.append(np.mean([abs(t - 1) < 1.96 * s for t, s in a])); cH.append(np.mean([abs(t - 1) < 1.96 * s for t, s in b]))
print(f"no heterogeneity: mean |gap| adaptive {np.mean(gA):.3f}, honest {np.mean(gH):.3f}; coverage adaptive {np.mean(cA):.3f}, honest {np.mean(cH):.3f}")

res = [run(het=1.0) for _ in range(100)]
pick = np.array([j == 0 for _, _, j in res]); hl = np.array([[b[0][0], b[1][0]] for _, b, _ in res])
print(f"heterogeneity: split on x1 in {pick.mean():.2f} of runs; honest leaf means then {hl[pick].mean(0).round(3)}")
print("Ex 7: E[max of two N(0,1)] =", round(float(np.maximum(*rng.normal(size=(2, 10 ** 6))).mean()), 3), "vs", round(1 / np.sqrt(np.pi), 3))

## Key takeaways

- Causal trees split to maximise differences in leaf effects; reusing the same data to estimate them overstates heterogeneity (winner's curse).
- Honesty separates search from estimation and restores valid leaf-level inference.
- Causal forests average honest trees into adaptive weights; check heterogeneity on held-out data with BLP and GATES.